K nearest neighbors - exercise 1
    Your first classification task will be to predict a patient's health status. Each patient was subjected to diagnostics, during which 6 biomechanical parameters related to the pelvis and lumbar spine were examined:

    pelvic incidence - also called: pelvic index
    pelvic tilt - tilt of the pelvis
    lumbar lordosis angle - angle of the lumbar lordosis
    sacral slope - angle of the base of the sacrum
    pelvic radius - radius of pelvis
    grade of spondylolisthesis - classification of spondylolisthesis.
    The patient's condition is assessed based on the above values - we detect whether the parient suffers from a disc herniation or spondylolisthesis. If the patient's condition does not indicate the presence of an abnormality, they are assigned the class Normal. Otherwise, the class Abnormal is assigned.

    The dataset consists of 310 observations, of which 100 are for patients without abnormal vertebral conditions, and 210 describe patients with disc herniation/spondylolisthesis.

1. Using the Pandas library, load the downloaded patient data file (lumbar_data.csv) into the lumbar_df variable. 
2. Display the first 10 observations in the dataset to see what type of data is in the file.
3. Check the data types of each variable, and verify the presence of empty values.
4. Perform data normalization to bring the data to a common scale.
5. Perform categorical encoding of the class variable. Assign the value of 0 to Normal, and the value of 1 to Abnormal.
6. Split the data into a training set and a test set in a 75/25 ratio. Set the random_state parameter to 42.
7. Import the appropriate class from the neighbors module of the Scikit-learn library, which you will use to build a k nearest neighbors model.
8. Create your own instance of the kNN model and train it on the created training set. Try to choose an appropriate value for the k parameter.
9. Use the trained model to perform a prediction on the test set.
10. Save the dataset from step 5 to a file named lumbar_normalized_df.csv.

In [41]:
# Import libraries and load data into DataFrame
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

lumbar_df = pd.read_csv("data/lumbar_data.csv")

# Check 10 first rows of DataFrame
lumbar_df.head(10)

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,class
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,Hernia
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,Hernia
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,Hernia
3,69.297008,24.652878,44.311238,44.644130,101.868495,11.211523,Hernia
4,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,Hernia
5,40.250200,13.921907,25.124950,26.328293,130.327871,2.230652,Hernia
6,53.432928,15.864336,37.165934,37.568592,120.567523,5.988551,Hernia
7,45.366754,10.755611,29.038349,34.611142,117.270067,-10.675871,Hernia
8,43.790190,13.533753,42.690814,30.256437,125.002893,13.289018,Hernia
9,36.686353,5.010884,41.948751,31.675469,84.241415,0.664437,Hernia


In [42]:
# Normalize numeric data
from sklearn.preprocessing import MinMaxScaler

lumbar_numeric_cols = lumbar_df.columns.drop("class")
scaler = MinMaxScaler()

lumbar_df[lumbar_numeric_cols] = scaler.fit_transform(lumbar_df[lumbar_numeric_cols])

In [43]:
# Categorical encoding for dependable variable

lumbar_df["class"] = lumbar_df["class"].replace({
    "Normal": 0, 
    "Hernia": 1,
    "Spondylolisthesis": 1
    })
lumbar_df["class"] = lumbar_df["class"].infer_objects(copy=False)
lumbar_df["class"].value_counts()

C:\Users\UrosVukmanovic\AppData\Local\Temp\ipykernel_29984\3402911838.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lumbar_df["class"] = lumbar_df["class"].replace({


class
1    210
0    100
Name: count, dtype: int64

In [47]:
# Split data into training and test dataset
X = lumbar_df.drop("class", axis=1)
y = lumbar_df["class"]

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.25,
                                                    random_state=42,
                                                    stratify=y)

# Initialize k nearest neighbors and fit training data
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Predict labels based on test data
y_pred = knn.predict(X_test)
print(f"KNN score:\t{knn.score(X_test, y_test)}")

KNN score:	0.8076923076923077


## What the model code does

The feature data (`X`) contains the six normalized biomechanical measurements, while the target (`y`) contains the encoded diagnosis: `0` for Normal and `1` for Abnormal.

`train_test_split` divides the observations into 75% training data and 25% test data. Setting `random_state=42` makes the split reproducible. `stratify=y` keeps the Normal/Abnormal class proportions approximately the same in both sets, which is useful because the classes are not equally represented.

The `KNeighborsClassifier` is initialized with `n_neighbors=5` and fitted on the training data. For each test observation, it finds the five nearest training observations and predicts the class most common among those neighbors. `y_pred` stores those predicted labels.

Finally, `knn.score(X_test, y_test)` calculates the fraction of test diagnoses predicted correctly. The recorded score is approximately **0.808 (80.8%)**. This is the model's accuracy on this particular test split; it does not mean every prediction is correct or guarantee the same performance on new data.